In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
import numpy as np
import pickle

#prepare
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(root='images', transform=transform)
print(f"Total images: {len(dataset)}")
print(f"Classes: {dataset.classes}")

targets = [label for _, label in dataset.samples]

train_indices, val_indices = train_test_split(
    np.arange(len(dataset)),
    test_size=0.2, # 20% for val
    stratify=targets,
    random_state=42
)

train_subset = Subset(dataset, train_indices)
val_subset = Subset(dataset, val_indices)

print(f"Training set size: {len(train_subset)}")
print(f"Validation set size: {len(val_subset)}")

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

# neural network
class FruitClassifier(nn.Module):
    def __init__(self):
        super(FruitClassifier, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1) # 3 input channels (r,g,b)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 16 * 16, 128)  #fully connected layer
        self.fc2 = nn.Linear(128, 9)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 32 * 16 * 16)  # flattens the tensor
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# init
model = FruitClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {running_loss / len(train_loader)}")

train_model(model, train_loader, criterion, optimizer)


def evaluate_model(model, val_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    print(f"Validation Accuracy: {accuracy * 100:.2f}%")

evaluate_model(model, val_loader)

# train a new model on the entire dataset to use all our data
print("Training a new model on the entire dataset...")
new_model = FruitClassifier().to(device)  # reinitialize
new_optimizer = optim.Adam(new_model.parameters(), lr=0.001)
full_loader = DataLoader(dataset, batch_size=32, shuffle=True)

train_model(new_model, full_loader, criterion, new_optimizer)

torch.save(new_model.state_dict(), 'model.pth')
print("New model saved as model.pth")

Total images: 359
Classes: ['apple', 'banana', 'cherry', 'chickoo', 'grapes', 'kiwi', 'mango', 'orange', 'strawberry']
Training set size: 287
Validation set size: 72
Epoch 1/10, Loss: 2.2078437540266247
Epoch 2/10, Loss: 2.0181718137529163
Epoch 3/10, Loss: 1.7391343514124553
Epoch 4/10, Loss: 1.5148286289638944
Epoch 5/10, Loss: 1.3521643280982971
Epoch 6/10, Loss: 1.2224702835083008
Epoch 7/10, Loss: 1.0985886851946514
Epoch 8/10, Loss: 0.9973683224784003
Epoch 9/10, Loss: 0.9303050107426114
Epoch 10/10, Loss: 0.8893369568718804
Validation Accuracy: 50.00%
Training a new model on the entire dataset...
Epoch 1/10, Loss: 2.2081693609555564
Epoch 2/10, Loss: 2.0331676503022513
Epoch 3/10, Loss: 1.7959968050320942
Epoch 4/10, Loss: 1.5944847464561462
Epoch 5/10, Loss: 1.3494855463504791
Epoch 6/10, Loss: 1.1665952603022258
Epoch 7/10, Loss: 1.1067130466302235
Epoch 8/10, Loss: 1.004206344485283
Epoch 9/10, Loss: 0.9224652995665868
Epoch 10/10, Loss: 0.8942203869422277
New model saved as 